In [ ]:
!pip install -q transformers datasets tensorflow scikit-learn sentencepiece

In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.52.4

Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 47.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.20.1
    Uninstalling huggingface_hub-1.20.1:
      Successfully uninstalled huggingface_hub-1.20.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from tf_keras.optimizers import Adam
from tf_keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("TF version:", tf.__version__)
print("GPU dispo:", tf.config.list_physical_devices('GPU'))


TF version: 2.20.0
GPU dispo: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv(
    "/content/drive/My Drive/Datasets/ShieldMind model/shieldmind_dataset2.csv",
    sep="\t",          # séparateur tabulation
    index_col=0        # la première colonne est l'index
)
df = df.dropna()
print("Shape brute:", df.shape)

df = df.drop_duplicates(subset=["comment_text"])
print("Shape après dédoublonnage:", df.shape)

print(df["class_name"].value_counts())
print(df.head())



Shape brute: (16000, 11)
Shape après dédoublonnage: (16000, 11)
class_name
violence          2000
clean             2000
self_harm         2000
sexual_content    2000
substances        2000
terrorism         2000
hate_speech       2000
vulgarity         2000
Name: count, dtype: int64
                                        comment_text      class_name  clean  \
0  Elle a reçu un coup de tête clairement violent...        violence      0   
1  Elle a reçu des coups franchement violents ava...        violence      0   
2  Ce repas était vraiment délicieux, merci. Séri...           clean      1   
3  Je pense sans arrêt à disparaître, ça devient ...       self_harm      0   
4  La photo qu'il a postée est jugée beaucoup tro...  sexual_content      0   

   violence  sexual_content  hate_speech  substances  self_harm  terrorism  \
0         1               0            0           0          0          0   
1         1               0            0           0          0          0   
2     

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit

encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["class_name"])
NUM_LABELS = len(encoder.classes_)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["template_id"]))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print("Classes:", encoder.classes_)
print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(f"NUM_LABELS: {NUM_LABELS}")

overlap = set(train_df["template_id"]) & set(test_df["template_id"])
print(f"Templates en commun train/test (doit être 0) : {len(overlap)}")

Classes: ['clean' 'hate_speech' 'self_harm' 'sexual_content' 'substances'
 'terrorism' 'violence' 'vulgarity']
Train: 12874 | Test: 3126
NUM_LABELS: 8
Templates en commun train/test (doit être 0) : 0


In [ ]:
MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

train_encodings = tokenizer(
    train_df["comment_text"].tolist(),
    truncation=True, padding="max_length",
    max_length=MAX_LEN, return_tensors="tf"
)
test_encodings = tokenizer(
    test_df["comment_text"].tolist(),
    truncation=True, padding="max_length",
    max_length=MAX_LEN, return_tensors="tf"
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
BATCH_SIZE = 16

class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weights = dict(enumerate(class_weights_arr))
print("Poids de classes:", class_weights)

train_sample_weights = train_df["label"].map(class_weights).values.astype("float32")

train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_df["label"].values.astype("int32"),
    train_sample_weights
)).shuffle(5000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    test_df["label"].values.astype("int32")
)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)



Poids de classes: {0: np.float64(1.0999658236500343), 1: np.float64(1.038225806451613), 2: np.float64(0.8905644714997233), 3: np.float64(0.9328985507246377), 4: np.float64(1.0217460317460318), 5: np.float64(0.9836491442542787), 6: np.float64(0.9477326266195524), 7: np.float64(1.133274647887324)}


In [ ]:
model = TFAutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    from_pt=True,
    num_labels=NUM_LABELS
)

# Backbone entraînable (pas de freeze)
optimizer = Adam(learning_rate=2e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer=optimizer, loss=loss, metrics=["accuracy"])
print("✅ Modèle compilé — backbone entraînable")




pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertForSequenceClassification: ['fit_denses.3.weight', 'fit_denses.1.weight', 'fit_denses.2.weight', 'fit_denses.4.weight', 'fit_denses.3.bias', 'fit_denses.0.bias', 'fit_denses.1.bias', 'fit_denses.4.bias', 'fit_denses.0.weight', 'fit_denses.2.bias']
- This IS expected if you are initializing TFBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classif

✅ Modèle compilé — backbone entraînable


In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6)
]

history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=40,
    callbacks=callbacks
)

Epoch 1/40
805/805 [==============================] - 110s 98ms/step - loss: 0.9804 - accuracy: 0.8208 - val_loss: 0.5731 - val_accuracy: 0.8474 - lr: 2.0000e-05
Epoch 2/40
805/805 [==============================] - 51s 63ms/step - loss: 0.0835 - accuracy: 0.9984 - val_loss: 0.7012 - val_accuracy: 0.8464 - lr: 2.0000e-05
Epoch 3/40
805/805 [==============================] - 62s 77ms/step - loss: 0.0270 - accuracy: 0.9995 - val_loss: 0.5857 - val_accuracy: 0.8602 - lr: 1.0000e-05


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import numpy as np
outputs = model.predict(test_dataset)

if hasattr(outputs, "logits"):
    logits = outputs.logits
elif isinstance(outputs, dict) and "logits" in outputs:
    logits = outputs["logits"]
else:
    logits = outputs

y_pred = np.argmax(logits, axis=1)
y_true = test_df["label"].values

print("F1 Macro :", f1_score(y_true, y_pred, average="macro"))
print(classification_report(y_true, y_pred, target_names=encoder.classes_))
print(confusion_matrix(y_true, y_pred))

196/196 [==============================] - 6s 25ms/step
F1 Macro : 0.8302408495727606
                precision    recall  f1-score   support

         clean       0.82      0.71      0.76       537
   hate_speech       0.90      1.00      0.95       450
     self_harm       0.98      0.64      0.77       193
sexual_content       0.94      0.75      0.83       275
    substances       0.71      0.97      0.82       425
     terrorism       0.92      0.75      0.83       364
      violence       0.65      0.74      0.69       302
     vulgarity       0.97      1.00      0.98       580

      accuracy                           0.85      3126
     macro avg       0.86      0.82      0.83      3126
  weighted avg       0.86      0.85      0.85      3126

[[380  42   2  12  65  23  12   1]
 [  0 450   0   0   0   0   0   0]
 [ 23   6 123   0   0   0  41   0]
 [ 30   0   0 206  22   0   0  17]
 [  0   0   0   1 413   0  11   0]
 [ 31   0   0   0   0 274  59   0]
 [  0   0   0   0  78   0 224

In [ ]:
loss, accuracy = model.evaluate(test_dataset)
print("Accuracy:", accuracy)

196/196 [==============================] - 5s 24ms/step - loss: 0.5731 - accuracy: 0.8474
Accuracy: 0.8474088311195374


In [ ]:
model.save_pretrained("shieldmind_tf")

tokenizer.save_pretrained("shieldmind_tf")

('shieldmind_tf/tokenizer_config.json',
 'shieldmind_tf/special_tokens_map.json',
 'shieldmind_tf/vocab.txt',
 'shieldmind_tf/added_tokens.json',
 'shieldmind_tf/tokenizer.json')

In [ ]:
# ==================== Fonction predict — modèle Keras (en mémoire) ====================
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="tf",
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )
    logits = model(**inputs).logits
    probs = tf.nn.softmax(logits, axis=-1)
    pred_id = int(tf.argmax(probs, axis=-1).numpy()[0])
    confidence = float(tf.reduce_max(probs).numpy())
    label = encoder.classes_[pred_id]   # déjà une string, ex: "clean"
    return label, f"{confidence:.1%}"


# ==================== Fonction predict — modèle TFLite exporté ====================
def predict_tflite(text):
    enc = tokenizer(
        text, truncation=True, padding="max_length",
        max_length=MAX_LEN, return_tensors="np"
    )
    input_ids = enc["input_ids"].astype(np.int32)
    attention_mask = enc["attention_mask"].astype(np.int32)
    token_type_ids = enc.get("token_type_ids", np.zeros_like(input_ids)).astype(np.int32)

    for detail in input_details:
        if "input_ids" in detail["name"]:
            interpreter.set_tensor(detail["index"], input_ids)
        elif "attention_mask" in detail["name"]:
            interpreter.set_tensor(detail["index"], attention_mask)
        else:
            interpreter.set_tensor(detail["index"], token_type_ids)

    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]["index"])
    pred_id = int(np.argmax(output))
    confidence = float(tf.nn.softmax(output, axis=-1).numpy().max())
    label = encoder.classes_[pred_id]   # déjà une string aussi, ex: "clean"
    return label, f"{confidence:.1%}"


# ==================== Jeu de test (labels corrects) ====================
tests = [
    ("The weather is beautiful today", "clean"),
    ("I enjoy reading books every evening", "clean"),
    ("Nous allons au marché demain matin", "clean"),
    ("J'adore cuisiner avec ma famille", "clean"),
    ("Learning Python is very interesting", "clean"),
    ("The meeting starts at nine o'clock", "clean"),
    ("Je regarde un documentaire sur les animaux", "clean"),
    ("We are planning our vacation for next month", "clean"),

    ("I will beat you if you come here again", "violence"),
    ("I'm going to break his arm", "violence"),
    ("Je vais te frapper", "violence"),
    ("On va le tabasser ce soir", "violence"),
    ("They threatened to kill everyone", "violence"),
    ("Nous allons attaquer ce groupe", "violence"),
    ("He wants to stab his neighbor", "violence"),
    ("Ils veulent incendier la maison", "violence"),

    ("These immigrants are ruining our country", "hate_speech"),
    ("I hate all people from that religion", "hate_speech"),
    ("Tous les étrangers sont des voleurs", "hate_speech"),
    ("Cette ethnie ne mérite aucun respect", "hate_speech"),
    ("People from that country are disgusting", "hate_speech"),
    ("Je déteste tous les musulmans", "hate_speech"),
    ("Nobody from that tribe should be trusted", "hate_speech"),
    ("Black people are all criminals", "hate_speech"),

    ("You are an idiot", "vulgarity"),
    ("Shut up you moron", "vulgarity"),
    ("Ferme ta gueule", "vulgarity"),
    ("Espèce de crétin", "vulgarity"),
    ("You're a useless loser", "vulgarity"),
    ("Quel imbécile tu fais", "vulgarity"),
    ("You're a piece of trash", "vulgarity"),
    ("Va te faire voir", "vulgarity"),

    ("Where can I buy cocaine?", "substances"),
    ("I need some heroin", "substances"),
    ("Où peut-on acheter de la drogue ?", "substances"),
    ("Il cherche de la cocaïne", "substances"),
    ("Who sells illegal drugs here?", "substances"),
    ("Je veux acheter du cannabis", "substances"),
    ("Do you know where to find meth?", "substances"),
    ("Il vend des substances illicites", "substances"),

    ("We must attack the government", "terrorism"),
    ("Let's bomb the building", "terrorism"),
    ("Nous allons faire exploser ce bâtiment", "terrorism"),
    ("Ils préparent une attaque contre les autorités", "terrorism"),
    ("The group is planning a terrorist attack", "terrorism"),
    ("Ils veulent poser une bombe", "terrorism"),
    ("Attack the civilians tomorrow", "terrorism"),
    ("Nous devons éliminer nos ennemis", "terrorism"),

    ("Send me nude pictures", "sexual_content"),
    ("Show me your naked body", "sexual_content"),
    ("Envoie-moi des photos intimes", "sexual_content"),
    ("Je veux voir ton corps nu", "sexual_content"),
    ("Let's have explicit video calls", "sexual_content"),
    ("Can you send intimate pictures?", "sexual_content"),
    ("Share your private photos with me", "sexual_content"),
    ("Envoie-moi une vidéo intime", "sexual_content"),
]


# ==================== Vérification des labels avant test ====================
expected_labels = set(t[1] for t in tests)
known_labels = set(encoder.classes_)
unknown = expected_labels - known_labels
if unknown:
    print("⚠️ Labels inconnus :", unknown)
else:
    print("✅ Tous les labels correspondent à encoder.classes_")


# ==================== Test du modèle Keras ====================
print("\n=== Modèle Keras (en mémoire) ===")
correct = 0
for text, expected in tests:
    predicted, confidence = predict(text)
    ok = "✅" if predicted == expected else "❌"
    if predicted == expected:
        correct += 1
    print(f"{ok} [{expected:15}] → {predicted:15} ({confidence}) | {text}")
print(f"\nScore Keras : {correct}/{len(tests)} = {correct/len(tests)*100:.1f}%")


✅ Tous les labels correspondent à encoder.classes_

=== Modèle Keras (en mémoire) ===
✅ [clean          ] → clean           (85.2%) | The weather is beautiful today
✅ [clean          ] → clean           (83.6%) | I enjoy reading books every evening
❌ [clean          ] → terrorism       (79.6%) | Nous allons au marché demain matin
✅ [clean          ] → clean           (72.3%) | J'adore cuisiner avec ma famille
✅ [clean          ] → clean           (67.5%) | Learning Python is very interesting
✅ [clean          ] → clean           (84.3%) | The meeting starts at nine o'clock
✅ [clean          ] → clean           (84.9%) | Je regarde un documentaire sur les animaux
✅ [clean          ] → clean           (84.5%) | We are planning our vacation for next month
✅ [violence       ] → violence        (86.1%) | I will beat you if you come here again
✅ [violence       ] → violence        (86.6%) | I'm going to break his arm
✅ [violence       ] → violence        (84.7%) | Je vais te frapper
❌ [viole

In [ ]:
MAX_LEN = 128

class ShieldMindModule(tf.Module):

    def __init__(self, model):
        super().__init__()
        self.model = model

    @tf.function(
        input_signature=[
            tf.TensorSpec([1, MAX_LEN], tf.int32, name="input_ids"),
            tf.TensorSpec([1, MAX_LEN], tf.int32, name="attention_mask"),
            tf.TensorSpec([1, MAX_LEN], tf.int32, name="token_type_ids"),
        ]
    )
    def __call__(self, input_ids, attention_mask, token_type_ids):

        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            training=False,
        )

        return {
            "logits": outputs.logits
        }

In [ ]:
export_module = ShieldMindModule(model)

In [ ]:
sample = tokenizer(
    "This is a test",
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN,
    return_tensors="tf"
)

token_type_ids = sample.get(
    "token_type_ids",
    tf.zeros_like(sample["input_ids"])
)

result = export_module(
    sample["input_ids"],
    sample["attention_mask"],
    token_type_ids
)

print(result["logits"])

tf.Tensor(
[[ 0.5122861  -0.59846896 -1.1137363   2.2206347  -0.28173095  0.512111
  -0.58526856 -0.37795645]], shape=(1, 8), dtype=float32)


In [ ]:
EXPORT_PATH = "/content/drive/My Drive/Models/ShieldMind/saved_model_export"

tf.saved_model.save(
    export_module,
    EXPORT_PATH
)

print("Export terminé :", EXPORT_PATH)

Export terminé : /content/drive/My Drive/Models/ShieldMind/saved_model_export


In [ ]:
loaded = tf.saved_model.load(EXPORT_PATH)

print(list(loaded.signatures.keys()))

['serving_default']


In [ ]:
infer = loaded.signatures["serving_default"]

print(infer.structured_input_signature)
print(infer.structured_outputs)

((), {'attention_mask': TensorSpec(shape=(1, 128), dtype=tf.int32, name='attention_mask'), 'input_ids': TensorSpec(shape=(1, 128), dtype=tf.int32, name='input_ids'), 'token_type_ids': TensorSpec(shape=(1, 128), dtype=tf.int32, name='token_type_ids')})
{'logits': TensorSpec(shape=(1, 8), dtype=tf.float32, name='logits')}


In [ ]:
import os
import numpy as np
import tensorflow as tf

SAVE_PATH = "/content/drive/My Drive/Models/ShieldMind/saved_model_export"

TFLITE_PATH = "/content/drive/My Drive/Models/ShieldMind/shieldmindv4.tflite"

In [ ]:
MAX_LEN = 128

def representative_dataset():

    limit = min(100, len(test_df))

    for i in range(limit):

        text = test_df.iloc[i]["comment_text"]

        enc = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="np"
        )

        token_type_ids = enc.get(
            "token_type_ids",
            np.zeros_like(enc["input_ids"])
        )

        yield {
            "input_ids": enc["input_ids"].astype(np.int32),
            "attention_mask": enc["attention_mask"].astype(np.int32),
            "token_type_ids": token_type_ids.astype(np.int32),
        }

In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model(SAVE_PATH)

# Quantification DYNAMIC RANGE : compresse les poids en int8,
# mais laisse les activations en float32 pendant l'inférence.
# Beaucoup plus stable pour les petits transformers que le full-int8.
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Pas de representative_dataset, pas de target_spec.supported_ops,
# pas de inference_input_type/output_type en int8 : on laisse le
# convertisseur choisir automatiquement quoi quantifier.

tflite_model = converter.convert()

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

print("Conversion réussie")
print(os.path.getsize(TFLITE_PATH) / (1024 * 1024), "MB")

Conversion réussie
13.918609619140625 MB


In [ ]:
try:

    tflite_model = converter.convert()

    with open(TFLITE_PATH, "wb") as f:
        f.write(tflite_model)

    print("Conversion réussie")

    print(
        os.path.getsize(TFLITE_PATH) / (1024 * 1024),
        "MB"
    )

except Exception as e:

    print(e)

Conversion réussie
13.918609619140625 MB


In [ ]:
interpreter = tf.lite.Interpreter(
    model_path=TFLITE_PATH
)

interpreter.allocate_tensors()

for inp in interpreter.get_input_details():

    print(inp["name"])
    print(inp["shape"])
    print(inp["dtype"])

print("---------------")

for out in interpreter.get_output_details():

    print(out["name"])
    print(out["shape"])
    print(out["dtype"])

serving_default_attention_mask:0
[  1 128]
<class 'numpy.int32'>
serving_default_input_ids:0
[  1 128]
<class 'numpy.int32'>
serving_default_token_type_ids:0
[  1 128]
<class 'numpy.int32'>
---------------
StatefulPartitionedCall:0
[1 8]
<class 'numpy.float32'>


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
print("Overlap train/test :", len(set(train_df.comment_text) & set(test_df.comment_text)))

Overlap train/test : 0


In [ ]:
import tensorflow as tf

interpreter = tf.lite.Interpreter(
    model_path="/content/drive/My Drive/Models/ShieldMind/shieldmindv4.tflite"
)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

In [ ]:
import tensorflow as tf

interpreter = tf.lite.Interpreter(
    model_path="/content/drive/My Drive/Models/ShieldMind/shieldmindv4.tflite"
)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

In [ ]:
def predict_tflite(text):
    enc = tokenizer(
        text, truncation=True, padding="max_length",
        max_length=128, return_tensors="np"
    )
    input_ids = enc["input_ids"].astype(np.int32)
    attention_mask = enc["attention_mask"].astype(np.int32)
    token_type_ids = enc.get("token_type_ids", np.zeros_like(input_ids)).astype(np.int32)

    for detail in input_details:
        if "input_ids" in detail["name"]:
            interpreter.set_tensor(detail["index"], input_ids)
        elif "attention_mask" in detail["name"]:
            interpreter.set_tensor(detail["index"], attention_mask)
        else:
            interpreter.set_tensor(detail["index"], token_type_ids)

    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]["index"])  # déjà float32
    pred = np.argmax(output)
    return pred, output

In [ ]:
import tensorflow as tf
from transformers import AutoTokenizer

SAVE_PATH = "/content/drive/My Drive/Models/ShieldMind/saved_model_export"
MAX_LEN = 128
MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
loaded = tf.saved_model.load(SAVE_PATH)
infer = loaded.signatures["serving_default"]

def predict_savedmodel(text):
    enc = tokenizer(
        text, truncation=True, padding="max_length",
        max_length=MAX_LEN, return_tensors="tf"
    )
    token_type_ids = enc.get("token_type_ids", tf.zeros_like(enc["input_ids"]))
    result = infer(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        token_type_ids=token_type_ids
    )
    # la clé de sortie peut être "logits" ou "output_0" selon la signature — on affiche les deux au cas où
    key = list(result.keys())[0]
    return result[key].numpy()

print("Phrase 1:", predict_savedmodel("I will kill you tomorrow."))
print("Phrase 2:", predict_savedmodel("I hate all black people."))
print("Phrase 3:", predict_savedmodel("The weather is nice today."))

Phrase 1: [[-0.80233884 -0.8120828  -0.3760072  -1.7359623  -1.3747497   2.6167526
   2.9828553  -0.65522337]]
Phrase 2: [[-0.769823   3.3764942 -0.848105  -1.1070993 -0.7252012  0.125267
  -1.6997741  1.3687949]]
Phrase 3: [[ 3.5951996   0.10255972  0.33192882 -0.12175851  0.6878534  -0.02824
  -1.1003171  -2.8443112 ]]


In [ ]:
pred1, logits1 = predict_tflite("I will kill you tomorrow.")
print("Classe 1:", encoder.inverse_transform([pred1])[0], logits1)

pred2, logits2 = predict_tflite("I hate all black people.")
print("Classe 2:", encoder.inverse_transform([pred2])[0], logits2)

pred3, logits3 = predict_tflite("The weather is nice today.")
print("Classe 3:", encoder.inverse_transform([pred3])[0], logits3)

Classe 1: violence [[-0.8097243  -0.8203319  -0.36561042 -1.7288108  -1.3776213   2.614268
   2.9823272  -0.65291625]]
Classe 2: hate_speech [[-0.84558356  3.3483028  -0.85029835 -1.0899583  -0.7228453   0.08826248
  -1.673433    1.452704  ]]
Classe 3: clean [[ 3.6090548   0.05843491  0.34846285 -0.09331664  0.6955944  -0.04589062
  -1.0861009  -2.8557684 ]]


In [ ]:
pred, logits = predict_tflite(
    "I will kill you tomorrow."
)

print("Classe prédite :", pred)
print("Logits :", logits)

Classe prédite : 6
Logits : [[-0.8097243  -0.8203319  -0.36561042 -1.7288108  -1.3776213   2.614268
   2.9823272  -0.65291625]]


In [ ]:
print(encoder.classes_)

['clean' 'hate_speech' 'self_harm' 'sexual_content' 'substances'
 'terrorism' 'violence' 'vulgarity']


In [ ]:
pred, logits = predict_tflite(
    "Let's build a bomb."
)

print("Classe prédite :", pred)
print("Logits :", logits)

Classe prédite : 5
Logits : [[-0.20158498  0.63125914 -0.48633844 -1.5775604  -2.326974    3.5599513
   1.1785522  -0.8123309 ]]


In [ ]:
pred, logits = predict_tflite(
    "Hello my friend, how are you?"
)

print("Classe prédite :", pred)
print("Logits :", logits)

Classe prédite : 0
Logits : [[ 2.987332   -0.8253023   2.326295   -0.43776268  0.3943956  -0.62035894
  -0.6469136  -2.4257095 ]]


In [ ]:
pred, logits = predict_tflite(
    "I hate all black people."
)

print("Classe :", encoder.inverse_transform([pred])[0])
print("Logits :", logits)

Classe : hate_speech
Logits : [[-0.84558356  3.3483028  -0.85029835 -1.0899583  -0.7228453   0.08826248
  -1.673433    1.452704  ]]


In [ ]:
pred, logits = predict_tflite(
    "I eat."
)

print("Classe :", encoder.inverse_transform([pred])[0])
print("Logits :", logits)

Classe : clean
Logits : [[ 2.6716967  -0.6870309  -0.42890498  1.0254118   2.5866907  -1.6066687
  -1.0777613  -1.9313152 ]]


In [ ]:
pred, logits = predict_tflite(
    "Hello everyone, have a nice day."
)

print("Classe :", encoder.inverse_transform([pred])[0])
print("Logits :", logits)

Classe : clean
Logits : [[ 3.6501768  -0.47668245  0.79382986  0.14513329  0.57479924 -0.16300146
  -0.8830038  -2.9194624 ]]


In [ ]:
pred, logits = predict_tflite(
    "You are stupid."
)

print("Classe :", encoder.inverse_transform([pred])[0])
print("Logits :", logits)

Classe : terrorism
Logits : [[ 0.20496275  0.98132724 -0.39578718 -1.9075559  -2.2942233   3.5933123
   0.9817962  -1.1652346 ]]


In [ ]:
pred, logits = predict_tflite(
    "I hate all muslims."
)

print("Classe :", encoder.inverse_transform([pred])[0])
print("Logits :", logits)

Classe : hate_speech
Logits : [[-0.28557265  3.4595022  -0.81302184 -1.2041254  -0.7751557   0.35693052
  -1.7700362   0.8123957 ]]


In [ ]:
pred, logits = predict_tflite(
    "I will kill you tomorrow."
)

print("Classe :", encoder.inverse_transform([pred])[0])
print("Logits :", logits)

Classe : violence
Logits : [[-0.8097243  -0.8203319  -0.36561042 -1.7288108  -1.3776213   2.614268
   2.9823272  -0.65291625]]


In [ ]:
pred, logits = predict_tflite(
    "Let's buy some cocaine."
)

print("Classe :", encoder.inverse_transform([pred])[0])
print("Logits :", logits)

Classe : substances
Logits : [[-0.24614292 -0.7975636  -1.2515798   0.3016831   3.352928   -2.2007246
   0.1617684   0.48937798]]


In [ ]:
common = set(train_df["comment_text"]) & set(test_df["comment_text"])
print("Nombre de textes communs :", len(common))

Nombre de textes communs : 0


In [ ]:
import tensorflow as tf
import numpy as np

MODEL_PATH = "/content/drive/My Drive/Models/ShieldMind/shieldmindv4.tflite"

# Chargement du modèle
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

print("="*70)
print("MODELE CHARGE AVEC SUCCES")
print("="*70)

# Entrées
input_details = interpreter.get_input_details()

print("\nINPUTS")
print("-"*70)

for i, inp in enumerate(input_details):
    print(f"Entrée {i}")
    print("Nom :", inp["name"])
    print("Shape :", inp["shape"])
    print("Type :", inp["dtype"])
    print("Quantization :", inp["quantization"])
    print()

# Sorties
output_details = interpreter.get_output_details()

print("\nOUTPUTS")
print("-"*70)

for i, out in enumerate(output_details):
    print(f"Sortie {i}")
    print("Nom :", out["name"])
    print("Shape :", out["shape"])
    print("Type :", out["dtype"])
    print("Quantization :", out["quantization"])
    print()

print("="*70)

# -------------------------------------------------------
# Création d'une entrée factice
# -------------------------------------------------------

for inp in input_details:

    shape = inp["shape"]
    dtype = inp["dtype"]

    if dtype == np.int32:
        fake = np.ones(shape, dtype=np.int32)

    elif dtype == np.int64:
        fake = np.ones(shape, dtype=np.int64)

    elif dtype == np.float32:
        fake = np.ones(shape, dtype=np.float32)

    elif dtype == np.int8:
        fake = np.ones(shape, dtype=np.int8)

    else:
        fake = np.zeros(shape, dtype=dtype)

    interpreter.set_tensor(inp["index"], fake)

# -------------------------------------------------------
# Inférence
# -------------------------------------------------------

print("Execution du modèle...")

interpreter.invoke()

print("OK")

print()

# -------------------------------------------------------
# Résultat
# -------------------------------------------------------

for out in output_details:

    result = interpreter.get_tensor(out["index"])

    print("Sortie brute :")
    print(result)

    print()

    if result.ndim == 2:

        probs = result[0]

        print("Valeurs :")

        for i, p in enumerate(probs):
            print(f"Classe {i} : {p}")

        print()

        pred = np.argmax(probs)

        print("Classe prédite :", pred)

print("="*70)

MODELE CHARGE AVEC SUCCES

INPUTS
----------------------------------------------------------------------
Entrée 0
Nom : serving_default_attention_mask:0
Shape : [  1 128]
Type : <class 'numpy.int32'>
Quantization : (0.0, 0)

Entrée 1
Nom : serving_default_input_ids:0
Shape : [  1 128]
Type : <class 'numpy.int32'>
Quantization : (0.0, 0)

Entrée 2
Nom : serving_default_token_type_ids:0
Shape : [  1 128]
Type : <class 'numpy.int32'>
Quantization : (0.0, 0)


OUTPUTS
----------------------------------------------------------------------
Sortie 0
Nom : StatefulPartitionedCall:0
Shape : [1 8]
Type : <class 'numpy.float32'>
Quantization : (0.0, 0)

Execution du modèle...
OK

Sortie brute :
[[-2.2199943  -1.1085774  -1.3012255   0.25095886  0.11180342  0.03005547
   2.2429307   1.5110025 ]]

Valeurs :
Classe 0 : -2.219994306564331
Classe 1 : -1.1085773706436157
Classe 2 : -1.3012255430221558
Classe 3 : 0.2509588599205017
Classe 4 : 0.11180341988801956
Classe 5 : 0.030055467039346695
Classe 6 

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
text = "I will kill you"

inputs = tokenizer(
    text,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="np"
)

# TensorFlow
tf_logits = model(**{
    "input_ids": tf.constant(inputs["input_ids"]),
    "attention_mask": tf.constant(inputs["attention_mask"]),
    "token_type_ids": tf.constant(inputs["token_type_ids"])
}).logits.numpy()

# TFLite
interpreter = tf.lite.Interpreter(model_path="/content/drive/My Drive/Models/ShieldMind/shieldmindv4.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()

interpreter.set_tensor(input_details[0]["index"], inputs["input_ids"].astype(np.int32))
interpreter.set_tensor(input_details[1]["index"], inputs["attention_mask"].astype(np.int32))
interpreter.set_tensor(input_details[2]["index"], inputs["token_type_ids"].astype(np.int32))

interpreter.invoke()

output = interpreter.get_tensor(interpreter.get_output_details()[0]["index"])

print("TensorFlow :", tf_logits)
print("TFLite     :", output)

print("\nDifférence maximale :", np.max(np.abs(tf_logits - output)))

TensorFlow : [[ 0.14169727  0.38546538 -0.30748478 -1.848492   -2.1166089   3.505433
   1.5335207  -1.2778375 ]]
TFLite     : [[-1.9094703   0.20847061 -1.3987144   1.7930108  -0.47096774 -0.3345616
  -0.24363302  2.1073081 ]]

Différence maximale : 3.8399947


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
for inp in input_details:
    name = inp["name"]

    if "input_ids" in name:
        interpreter.set_tensor(
            inp["index"],
            inputs["input_ids"].astype(np.int32)
        )

    elif "attention_mask" in name:
        interpreter.set_tensor(
            inp["index"],
            inputs["attention_mask"].astype(np.int32)
        )

    elif "token_type_ids" in name:
        interpreter.set_tensor(
            inp["index"],
            inputs["token_type_ids"].astype(np.int32)
        )